# mfs-agent-security: benchmark run on Kaggle

**Before running:** in the right panel, set **Accelerator: GPU T4 x2** and turn **Internet on**. Check this again after importing a new version of the notebook: an import can reset the accelerator. The first cell stops the run at once if the session has no GPUs.

This version has two parts. Choose one with `PART` in section 3 and run the notebook once per part (separate Kaggle runs, each with its own `results.zip`). Parts A and B are done; part C is left.

- **Part A (about 4 to 5 hours):**
  - The **wallet-tool rerun**: the main set's 37 tool-description attacks (`bench/cases_wallet_tool`, the same cases with the same ids) on the five open models, with `none`, on today's pinned vLLM. Pairing them with the third-party runs of 3 November shows whether the gap between the two sets came from the older, unrecorded vLLM version.
  - Then **Qwen3-8B** and **Qwen3-14B-AWQ**, a newer open model family. Each runs on the main 302 cases, the 24 message-payee cases and the 37 third-party cases, with `none` and `provenance-amount`.
- **Part B (about 4 to 6 hours):** **Qwen3-32B-AWQ** on the same three case sets with the same two defences.
- **Part C (about 2 to 3 hours):** **Qwen3-14B-AWQ** alone, on the same three case sets with the same two defences. In part A (23 November) it did not start: its download stopped with "Read-only file system" after six other models had been downloaded in the same session, most likely because the disk was full. Run on its own, as the 32B model was in part B, it has the disk to itself. The notebook now also deletes each model's downloaded files once its runs are done.

Qwen3 runs with its thinking mode off (`enable_thinking: false`), so it answers like the other models. Results go to `results_wallet_tool/`, `results/`, `results_message_payee/` and `results_third_party/`.

**Time:** the times above are estimates. Downloading and starting each model takes 5 to 20 minutes (the 32B model the longest). Use **Save Version → Save & Run All (Commit)** and let it run in the background. Results are saved after every run, so if Kaggle stops the session, the finished runs are kept in `results.zip`. If the model cannot start (for example, not enough GPU memory, or the model is not on Hugging Face under that name) or cannot call tools, the notebook says so, saves the server log in `results.zip`, and skips it.

In [ ]:
# Stop at once if this session has no GPUs: vLLM cannot start without them.
import shutil, subprocess
gpus = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout if shutil.which("nvidia-smi") else ""
print(gpus or "No GPU found in this session.")
assert gpus.count("GPU ") >= 2, ("This session has no GPUs. In the right panel set Accelerator to 'GPU T4 x2' "
                                 "(and check that weekly GPU quota is left), then run again.")

## 1. Get the project and run its tests

In [ ]:
!git clone -q https://github.com/s4m404/mfs-agent-security.git
%cd mfs-agent-security
!pip install -q -r requirements.txt
!python -m pytest -q

## 2. Install vLLM

In [ ]:
# vLLM is pinned to 0.32.0, the newest version when the last runs worked on these T4s (23 October
# 2026); newer versions are untested here. run_env.json records the version each run used.
!pip install -q vllm==0.32.0
# Kaggle's pre-installed torchaudio is built for a different CUDA version than the
# PyTorch that vLLM installs, which crashes vLLM at startup. Text models never use
# audio, so remove it.
!pip uninstall -y -q torchaudio
# Check that vLLM finds the GPUs and loads its GPU code (without a GPU it fails with "Failed to infer
# device type"); this stops here with the real error if not. vLLM may also print "Failed to import
# from vllm._C": that is harmless, since newer vLLM versions keep their GPU code under another name.
# The versions are saved in run_env.json (in results.zip), so every run records what it ran on.
import json, time
check = subprocess.run(["python", "-c", (
    "import json, platform, torch, vllm, vllm.entrypoints.cli.main\n"
    "from vllm.platforms import current_platform  # finds the GPUs and loads vLLM's GPU code\n"
    "assert current_platform.is_cuda(), 'vLLM finds no CUDA GPU'\n"
    "print(json.dumps({'python': platform.python_version(), 'vllm': vllm.__version__, 'torch': torch.__version__,\n"
    "                  'cuda': torch.version.cuda,\n"
    "                  'gpus': [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]}))")],
    capture_output=True, text=True)
# vLLM writes its log to stdout, so on failure show both: the real error is at the end of stderr.
print(check.stdout if check.returncode == 0 else check.stdout[-2000:] + check.stderr[-3000:])
assert check.returncode == 0, "vLLM cannot use the GPUs. See the error above."
RUN_ENV = json.loads(check.stdout.strip().splitlines()[-1])
assert len(RUN_ENV["gpus"]) >= 2, "PyTorch sees fewer than 2 GPUs: set Accelerator to GPU T4 x2."
RUN_ENV.update(date=time.strftime("%Y-%m-%d"), models={})
json.dump(RUN_ENV, open("run_env.json", "w"), indent=1)

## 3. Settings
Edit these lists to change what runs. Each model runs once per defence. Each model needs the vLLM *tool parser* that matches how it writes tool calls.

In [ ]:
PART = "C"   # "A": wallet-tool rerun + Qwen3-8B and Qwen3-14B-AWQ; "B": Qwen3-32B-AWQ; "C": Qwen3-14B-AWQ alone

MODELS = {  # model: vLLM tool parser
    "Qwen/Qwen2.5-3B-Instruct": "hermes",
    "Qwen/Qwen2.5-7B-Instruct": "hermes",
    "Qwen/Qwen2.5-14B-Instruct-AWQ": "hermes",
    "NousResearch/Hermes-3-Llama-3.1-8B": "hermes",    # second family (Llama 3.1 8B)
    "Qwen/Qwen2.5-32B-Instruct-AWQ": "hermes",         # larger model, 4-bit quantised to fit the T4s
    # "ibm-granite/granite-3.3-8b-instruct": "granite",  # left out: tool calls came out as plain text
    "Qwen/Qwen3-8B": "hermes",                         # newer family (Qwen3), 16-bit
    "Qwen/Qwen3-14B-AWQ": "hermes",                    # 4-bit quantised
    "Qwen/Qwen3-32B-AWQ": "hermes",                    # 4-bit quantised
}
# Extra vLLM settings for models that need them. The 32B models only just fit
# the two T4s, so they may use more of the GPU memory and run at most 8 cases at once.
EXTRA_ARGS = {
    "Qwen/Qwen2.5-32B-Instruct-AWQ": ["--gpu-memory-utilization", "0.95", "--max-num-seqs", "8"],
    "Qwen/Qwen3-32B-AWQ": ["--gpu-memory-utilization", "0.95", "--max-num-seqs", "8"],
}
# Extra API settings sent with every request. Qwen3 thinks aloud by default; turn that off.
NO_THINKING = {"chat_template_kwargs": {"enable_thinking": False}}
EXTRA_BODY = {m: NO_THINKING for m in MODELS if m.startswith("Qwen/Qwen3-")}

# What each part runs: model -> list of (cases folder, output folder, defences).
OPEN_MODELS = ["Qwen/Qwen2.5-3B-Instruct", "Qwen/Qwen2.5-7B-Instruct", "Qwen/Qwen2.5-14B-Instruct-AWQ",
               "NousResearch/Hermes-3-Llama-3.1-8B", "Qwen/Qwen2.5-32B-Instruct-AWQ"]
NEW_MODEL_SETS = [
    ("bench/cases", "results", ["none", "provenance-amount"]),
    ("bench/cases_message_payee", "results_message_payee", ["none", "provenance-amount"]),
    ("bench/cases_third_party", "results_third_party", ["none", "provenance-amount"]),
]
PLANS = {
    "A": {**{m: [("bench/cases_wallet_tool", "results_wallet_tool", ["none"])] for m in OPEN_MODELS},
          "Qwen/Qwen3-8B": NEW_MODEL_SETS,
          "Qwen/Qwen3-14B-AWQ": NEW_MODEL_SETS},
    "B": {"Qwen/Qwen3-32B-AWQ": NEW_MODEL_SETS},
    "C": {"Qwen/Qwen3-14B-AWQ": NEW_MODEL_SETS},
}
PLAN = PLANS[PART]
OUT_FOLDERS = "run_env.json results results_adaptive results_message_payee results_third_party results_wallet_tool"
REPEATS = 1
WORKERS = 8   # cases run at the same time; set to 1 to run one by one

## 4. Helpers: start and stop the model server

In [ ]:
import subprocess, time, urllib.request, json, glob, os, signal, shutil
from openai import OpenAI

def start_server(model, parser, extra=()):
    proc = subprocess.Popen(
        ["vllm", "serve", model,
         "--dtype", "half",                 # T4 GPUs do not support bfloat16
         "--tensor-parallel-size", "2",
         "--max-model-len", "8192",
         "--enable-auto-tool-choice", "--tool-call-parser", parser,
         "--port", "8000", *extra],
        stdout=open("vllm.log", "w"), stderr=subprocess.STDOUT, start_new_session=True)
    for i in range(180):  # up to 30 minutes (large models take longer to download)
        try:
            urllib.request.urlopen("http://localhost:8000/v1/models", timeout=5)
            print(f"{model}: server ready after about {i * 10} seconds")
            return proc
        except Exception:
            if proc.poll() is not None:
                break
            time.sleep(10)
    print(f"{model}: server did not start. Last lines of vllm.log:")
    print(open("vllm.log").read()[-3000:])
    os.makedirs("results", exist_ok=True)  # keep the log in results.zip so the cause can be checked later
    with open(f"results/vllm_failed_{model.replace('/', '_')}.log", "w") as f:
        f.write(open("vllm.log").read())
    stop_server(proc)
    return None

def stop_server(proc):
    if proc and proc.poll() is None:
        os.killpg(os.getpgid(proc.pid), signal.SIGTERM)
        try:
            proc.wait(timeout=60)
        except Exception:
            os.killpg(os.getpgid(proc.pid), signal.SIGKILL)
    time.sleep(15)  # let the GPUs free their memory

def delete_model_files(model):
    """Free the disk once a model's runs are done (each model takes 6 to 20 GB)."""
    path = os.path.expanduser(f"~/.cache/huggingface/hub/models--{model.replace('/', '--')}")
    shutil.rmtree(path, ignore_errors=True)
    print(f"{model}: downloaded files {'deleted' if not os.path.exists(path) else 'could not be deleted'}")
    print(subprocess.run(["df", "-h", os.path.expanduser("~/.cache"), "/kaggle/working"], capture_output=True, text=True).stdout)

def tool_call_check(model, extra_body=None):
    client = OpenAI(base_url="http://localhost:8000/v1", api_key="not-needed")
    resp = client.chat.completions.create(
        model=model, temperature=0, extra_body=extra_body,
        messages=[{"role": "user", "content": "What is my wallet balance?"}],
        tools=[{"type": "function", "function": {
            "name": "check_balance", "description": "Return the user's balance.",
            "parameters": {"type": "object", "properties": {}}}}])
    ok = bool(resp.choices[0].message.tool_calls)
    print(f"{model}: tool calling {'works' if ok else 'DOES NOT WORK, skipping this model'}")
    if ok and "<think>" in (resp.choices[0].message.content or ""):
        print(f"{model}: the reply contains thinking text; check EXTRA_BODY. Skipping this model.")
        ok = False
    return ok

def record_revision(model):
    """Save the exact model version (the Hugging Face commit) in run_env.json."""
    try:
        from huggingface_hub import HfApi
        RUN_ENV["models"][model] = HfApi().model_info(model).sha
    except Exception as exc:  # no network or no such field: say so instead of failing the run
        RUN_ENV["models"][model] = f"unknown ({type(exc).__name__})"
    json.dump(RUN_ENV, open("run_env.json", "w"), indent=1)

## 5. Run everything
Each model in the chosen part: start server, check tool calling, run every case set and defence, stop server.

In [ ]:
for model, case_sets in PLAN.items():
    parser = MODELS[model]
    print(f"\n########## {model} ##########")
    record_revision(model)
    proc = start_server(model, parser, EXTRA_ARGS.get(model, ()))
    if proc is None:
        delete_model_files(model)
        continue
    extra_body = EXTRA_BODY.get(model)
    extra_flag = f"--extra-body '{json.dumps(extra_body)}'" if extra_body else ""
    try:
        if not tool_call_check(model, extra_body):
            continue
        for cases, out, defences in case_sets:
            for defence in defences:
                print(f"\n===== {model} | {cases} | defence={defence} =====")
                !python scripts/run_bench.py --model {model} --base-url http://localhost:8000/v1 --cases {cases} --out {out} --defence {defence} --repeats {REPEATS} --workers {WORKERS} {extra_flag} 2>&1 | grep -v "HTTP Request" | tail -25
                !zip -qr /kaggle/working/results.zip {OUT_FOLDERS}
    finally:
        stop_server(proc)
        delete_model_files(model)

## 6. Summary of all runs
`results.zip` in the Output panel has everything. Download it, unpack it next to the repository, and add the run folders to the `kaggle-results` branch (one dated folder per Kaggle run, for example `YYYY-MM-DD_wallet_tool_qwen3_part_a/`).

In [ ]:
!zip -qr /kaggle/working/results.zip {OUT_FOLDERS}
for p in sorted(glob.glob("results*/*/summary.md")):
    print(p)
    print(open(p).read().split("## Attack success by injection language")[0])